In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

In [2]:
base_path = r'c:\Users\haruka\Desktop\AI수업\mcp-month-2\data'

In [3]:
file_path = os.path.join(base_path, 'wine.csv')
wine_df = pd.read_csv(file_path)

In [4]:
data = wine_df[['alcohol', 'sugar', 'pH']].to_numpy()
target = wine_df['class'].to_numpy()

In [7]:
from sklearn.model_selection import train_test_split

train_input, test_input, train_target, test_target =\
train_test_split(data, target, test_size=0.2, random_state=42)

In [8]:
sub_input, val_input, sub_target, val_target =\
train_test_split(
  train_input, train_target, test_size=0.2, random_state=42
)

In [9]:
print(train_input.shape)
print(sub_input.shape)
print(val_input.shape)
print(test_input.shape)

(5197, 3)
(4157, 3)
(1040, 3)
(1300, 3)


In [11]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42)
dt.fit(sub_input, sub_target);

In [12]:
print(dt.score(sub_input, sub_target))
print(dt.score(val_input, val_target))

0.9971133028626413
0.864423076923077


In [13]:
from sklearn.model_selection import cross_validate

In [14]:
scores = cross_validate(dt, train_input, train_target)

In [15]:
scores

{'fit_time': array([0.00949788, 0.00806355, 0.00715971, 0.00762177, 0.0066762 ]),
 'score_time': array([0.00186372, 0.00161457, 0.00136423, 0.00143671, 0.00190616]),
 'test_score': array([0.86923077, 0.84615385, 0.87680462, 0.84889317, 0.83541867])}

In [16]:
print(np.mean(scores['test_score']))

0.855300214703487


In [18]:
from sklearn.model_selection import StratifiedKFold

In [20]:
splitter = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_validate(dt, train_input, train_target, cv=splitter)
print(np.mean(scores['test_score'])) # 0.857...

0.8574181117533719


In [22]:
from sklearn.model_selection import GridSearchCV

In [21]:
DecisionTreeClassifier().get_params()

{'ccp_alpha': 0.0,
 'class_weight': None,
 'criterion': 'gini',
 'max_depth': None,
 'max_features': None,
 'max_leaf_nodes': None,
 'min_impurity_decrease': 0.0,
 'min_samples_leaf': 1,
 'min_samples_split': 2,
 'min_weight_fraction_leaf': 0.0,
 'monotonic_cst': None,
 'random_state': None,
 'splitter': 'best'}

In [23]:
min_impurity_decrease = [0.0001, 0.0002, 0.0003, 0.0004, 0.0005]
params = {'min_impurity_decrease': min_impurity_decrease}

In [24]:
gs = GridSearchCV(DecisionTreeClassifier(random_state=42), params, n_jobs=-1, cv=5)

In [ ]:
gs.fit(train_input, train_target);

In [27]:
print("최적 점수 : {:.3f}".format(gs.best_score_))
print("최적 하이퍼 파라미터 : ", gs.best_params_)

최적 점수 : 0.868
최적 하이퍼 파라미터 :  {'min_impurity_decrease': 0.0001}


In [28]:
dt = gs.best_estimator_

In [29]:
print(dt.score(train_input, train_target))

0.9615162593804117


In [30]:
print(gs.cv_results_)

{'mean_fit_time': array([0.02935386, 0.01702642, 0.00949492, 0.00821209, 0.0079978 ]), 'std_fit_time': array([0.00257202, 0.00800692, 0.00118053, 0.00043903, 0.00080991]), 'mean_score_time': array([0.00257983, 0.00240102, 0.00246882, 0.00217595, 0.00196552]), 'std_score_time': array([0.00025867, 0.00016367, 0.00064884, 0.00015909, 0.0002693 ]), 'param_min_impurity_decrease': masked_array(data=[0.0001, 0.0002, 0.0003, 0.0004, 0.0005],
             mask=[False, False, False, False, False],
       fill_value=1e+20), 'params': [{'min_impurity_decrease': 0.0001}, {'min_impurity_decrease': 0.0002}, {'min_impurity_decrease': 0.0003}, {'min_impurity_decrease': 0.0004}, {'min_impurity_decrease': 0.0005}], 'split0_test_score': array([0.86923077, 0.87115385, 0.86923077, 0.86923077, 0.86538462]), 'split1_test_score': array([0.86826923, 0.86346154, 0.85961538, 0.86346154, 0.86923077]), 'split2_test_score': array([0.8825794 , 0.87680462, 0.87584216, 0.88161694, 0.8825794 ]), 'split3_test_score': arr

In [31]:
pd.DataFrame(gs.cv_results_)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_min_impurity_decrease,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.029354,0.002572,0.002580,0.000259,0.0001,{'min_impurity_decrease': 0.0001},0.869231,0.868269,0.882579,0.867180,0.853705,0.868193,0.009154,1
1,0.017026,0.008007,0.002401,0.000164,0.0002,{'min_impurity_decrease': 0.0002},0.871154,0.863462,0.876805,0.854668,0.856593,0.864536,0.008437,5
2,0.009495,0.001181,0.002469,0.000649,0.0003,{'min_impurity_decrease': 0.0003},0.869231,0.859615,0.875842,0.850818,0.869105,0.864922,0.008745,4
3,0.008212,0.000439,0.002176,0.000159,0.0004,{'min_impurity_decrease': 0.0004},0.869231,0.863462,0.881617,0.848893,0.875842,0.867809,0.011260,2
4,0.007998,0.000810,0.001966,0.000269,0.0005,{'min_impurity_decrease': 0.0005},0.865385,0.869231,0.882579,0.849856,0.871030,0.867616,0.010570,3
